In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import cv2, os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

In [3]:
DATASET_DIR = '/content/drive/MyDrive/data'

class_names = ['23-50277-1', '23-50254-1', '22-48133-2_', '22-48021-2', '22-46887-1']
IMG_SIZE = 128

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

In [4]:
def load_faces(base_dir):
    X, y = [], []
    for label, person in enumerate(class_names):
        person_dir = os.path.join(base_dir, person)
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)
            if len(faces) == 0:
                continue

            x, y_, w, h = faces[0]
            face = img[y_:y_+h, x:x+w]
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            X.append(face)
            y.append(label)

    return np.array(X), np.array(y)

X, Y = load_faces(DATASET_DIR)
print("Dataset Shape:", X.shape, Y.shape)

X, Y = shuffle(X, Y, random_state=42)

X = X.astype('float32') / 255.0


Dataset Shape: (104, 128, 128, 3) (104,)


In [5]:
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

inputs = layers.Input(shape=(128, 128, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(5, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
model.fit(X, Y, epochs=30, batch_size=16, validation_split=0.2)
model.save("AttendanceSystem_transfer_learning.keras")


Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 5s/step - accuracy: 0.2820 - loss: 2.0203 - val_accuracy: 0.7619 - val_loss: 0.7563
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6664 - loss: 0.7937 - val_accuracy: 0.8571 - val_loss: 0.4533
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9012 - loss: 0.3551 - val_accuracy: 0.9048 - val_loss: 0.3788
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9136 - loss: 0.2465 - val_accuracy: 0.9048 - val_loss: 0.2819
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9694 - loss: 0.1420 - val_accuracy: 0.9048 - val_loss: 0.1875
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9672 - loss: 0.1315 - val_accuracy: 0.9524 - val_loss: 0.1482
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9896 - loss: 0.0877 - val_accuracy: 0.9524 - val_loss: 0.1392
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9762 - loss: 0.0801 - val_accuracy: 0.9048 - val_loss: 0.1737
E